# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama-3.1-8b-instant'
openai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Respond ONLY with JSON in this exact format:
{{
    "links": [
        {{"type": "about page", "url": "https://full-url-here.com/about"}},
        {{"type": "careers page", "url": "https://full-url-here.com/careers"}}
    ]
}}

Links (some might be relative links):
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Respond ONLY with JSON in this exact format:
{
    "links": [
        {"type": "about page", "url": "https://full-url-here.com/about"},
        {"type": "careers page", "url": "https://full-url-here.com/careers"}
    ]
}

Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'careers page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama-3.1-8b-instant
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'careers page', 'url': 'https://edwarddonner.com/'},
  {'type': 'curriculum or courses page',
   'url': 'https://edwarddonner.com/curriculum/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'contact/press', 'url': 'mailto:press@huggingface.co'},
  {'type': 'community/forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status updates', 'url': 'https://status.huggingface.co/'},
  {'type': 'community/discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'social/twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social/linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        # Skip non-http links
        if not link["url"].startswith("http"):
            continue
        try:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
        except Exception as e:
            print(f"Skipping {link['url']} - {e}")
            continue
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
4 days ago
•
158k
•
735
Zyphra/ZAYA1-8B
Updated
1 day ago
•
66.1k
•
449
openbmb/MiniCPM-V-4.6
Updated
about 19 hours ago
•
393
HiDream-ai/HiDream-O1-Image
Updated
about 1 hour ago
•
3.42k
•
271
deepseek-ai/DeepSeek-V4-Pro
Updated
7 days ago
•
2.02M
•
3.89k
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.1k
Wan2.2 14B Fast Preview
🐌
1.1k
generate a video from an image with a text prompt
Running
143
The ultimate guide to RL environments: building and scaling them in the LLM era
📝
143
Building

In [26]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt_2 = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\n4 days ago\n•\n158k\n•\n735\nZyphra/ZAYA1-8B\nUpdated\n1 day ago\n•\n66.1k\n•\n449\nopenbmb/MiniCPM-V-4.6\nUpdated\nabout 19 hours ago\n•\n393\nHiDream-ai/HiDream-O1-Image\nUpdated\nabout 1 hour ago\n•\n3.42k\n•\n271\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n7 days ago\n•\n2.02M\n•\n3.89k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n1.1k\nWan2.2 14B

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 9 relevant links


**Welcome to Hugging Face: The AI Community Building the Future**

At Hugging Face, we believe that the future of Artificial Intelligence lies in collaboration and open-source innovation. Our platform empowers a vast community of machine learning engineers, scientists, and developers to create, share, and experiment with cutting-edge models, datasets, and applications.

**Core Values**

* **Collaboration**: Our platform is designed to facilitate collaboration and knowledge-sharing among the AI community.
* **Open-Source**: We believe in the power of open-source innovation and empower our users to use, modify, and share our models, datasets, and applications.
* **Innovation**: We encourage experimentation and innovation, providing our users with the tools and resources they need to push the boundaries of AI.

**Our Offerings**

* **Models**: Browse over 2 million pre-trained models, from text generation to image recognition, and use them in your applications.
* **Datasets**: Explore our vast collection of datasets, covering a wide range of topics and modalities, from text to images, video, audio, and 3D.
* **Spaces**: Create collaborative spaces with other users to experiment, test, and refine your models and applications.

**Who We Are**

Hugging Face is a fast-growing community of passionate individuals working together to create a future where AI is open, accessible, and beneficial to all. Our company is at the heart of the AI revolution, driven by our talented science team exploring the edge of tech.

**Join Our Community**

* **Explore our documentation**: Learn how to use our platform, models, and datasets to create your own AI applications.
* **Join our forums**: Engage with our community, ask questions, and share your experiences.
* **Find a career**: Explore our current openings and join our team of innovators and experts.

**Contact Us**

* **Website**: [huggingface.com](http://huggingface.com)
* **Twitter**: @huggingface
* **LinkedIn**: linkedin.com/company/huggingface
* **GitHub**: github.com/huggingface

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 4 relevant links


**Hugging Face Brochure**

**Welcome to Hugging Face**

Hugging Face is the collaboration platform for the machine learning community. Our mission is to enable the next generation of machine learning engineers, scientists, and end-users to create, share, and experiment with open-source ML models, datasets, and applications.

**Unlocking AI Potential**

Our platform is built on the idea that collaboration is key to unlocking the full potential of artificial intelligence. With Hugging Face, you can:

 * **Explore AI Apps**: Browse over 2 million pre-trained models and explore AI applications across various domains.
 * **Discover and Share**: Share your models, datasets, and applications with the community and discover new AI capabilities.
 * **Collaborate**: Host and collaborate on unlimited public models, datasets, and applications in a secure and accessible environment.

**The Power of HF Open Source Stack**

Our open-source stack is designed to accelerate your machine learning journey. With Hugging Face, you can:

 * **Move Faster**: Leverage our pre-built models, datasets, and applications to speed up your development.
 * **Explore All Modalities**: Develop and experiment with AI models across text, image, video, audio, and 3D modalities.
 * **Build Your Portfolio**: Share your work with the world and build your machine learning profile.

**Career Opportunities**

We are a fast-growing company with exciting career opportunities in areas such as:

 * **AI Engineering**: Help build and maintain our AI models and platforms.
 * **Product Management**: Drive the development of our products and features.
 * **Marketing**: Help spread the word about Hugging Face across the globe.

**Company Culture**

At Hugging Face, we value:

 * **Collaboration**: We believe that collaboration is key to unlocking AI potential.
 * **Innovation**: We encourage experimentation and innovation in all aspects of our platform.
 * **Inclusion**: We strive to create a welcoming and inclusive environment for all members of our community.

**Join the Hugging Face Journey**

Whether you're a seasoned AI expert or just starting out in machine learning, we invite you to join our community and explore the vast possibilities of Hugging Face.

In [28]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 8 relevant links


**Welcome to Hugging Face: The Heart of AI Revolution**

At Hugging Face, we're not just a company - we're a community of passionate individuals who believe in building an open and ethical AI future together. We're a platform where machine learning enthusiasts, engineers, and scientists collaborate, share, and innovate. Come and join us on this exciting journey!

**What We Do**

Hugging Face is a collaboration platform for the machine learning community. Our core products are:

* **Hugging Face Hub**: A central place where you can share, explore, discover, and experiment with open-source machine learning models, datasets, and applications.
* **Spaces**: An online environment where you can run, manage, and monitor your ML projects in a collaborative way.
* **Datasets**: A collection of curated and community-driven datasets for various modalities, including text, image, video, and more.

**Culture & Community**

At Hugging Face, we're committed to building a friendly and inclusive community. We believe in:

* **Open Source**: Share, collaborate, and learn from each other.
* **Innovation**: Explore the edge of tech and push the boundaries of AI.
* **Inclusion**: Everyone's welcome to contribute and learn.

**Customers & Partners**

Our community includes:

* **Researchers**: From top universities and research institutions.
* **Engineers**: From leading tech companies and startups.
* **Artists**: Using our platforms to create innovative artistic experiences.

**Careers**

Looking for a new challenge? Check out our current openings and join our team!

**What We're Known For**

* **Transformers**: Our pioneering work on transformer-based models has revolutionized the field of NLP and beyond.
* **Open-Source**: Our commitment to open-source has enabled the community to build, share, and learn from each other.

**Get Started**

Explore our platforms, browse our community blog, and don't hesitate to reach out. Together, let's build the future of AI!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>